### RAG Project - Mental Help Assistant
_Made by Sofia Kurakova_

### **1. Data Preparation - Extract text from PDF**

In [ ]:
pip install seaborn transformers pymilvus sentence-transformers huggingface-hub langchain_community langchain-text-splitters pypdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader1 = PyPDFLoader("DSM 5 TR.pdf")
loader2 = PyPDFLoader("Fundamentals-of-Psychological-Disorders.pdf")
docs1 = loader1.load()
docs2 = loader2.load()
all_docs = docs1+docs2
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(all_docs)

# loader = PyPDFLoader("Fundamentals-of-Psychological-Disorders.pdf")
# docs = loader.load()
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# chunks = text_splitter.split_documents(docs)
text_lines = [chunk.page_content for chunk in chunks]

### **2. Embedding a sentence Generation**

In [ ]:
pip install --upgrade jupyter ipywidgets

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

s = text_lines[0]
e = embedding_model.encode([s])

### **3. Creating a Milvus data collection**

In [ ]:
from pymilvus import MilvusClient

milvus_client = MilvusClient(uri="./rag_project.db") #next time the my_project database is already created --> don't use "create_collection"

collection_name = "mental_health_rag"

embedding_dim = e.shape[1]

if collection_name not in milvus_client.list_collections():
    milvus_client.create_collection(
        collection_name=collection_name,
        dimension=embedding_dim, # The size of the embedding
        metric_type="IP", # Inner product distance
        consistency_level="Strong", # Strong consistency level
    )
    print(f"Collection '{collection_name}' created.")
else:
    print(f"Collection '{collection_name}' already exists. Skipping creation.")


In [ ]:
def emb_text(line):
    return embedding_model.encode(line)

In [ ]:
data = []

# In the following example, emb_text is a function that needs to be written, based on
# an embedding model
for i, line in enumerate(text_lines):
    data.append({"id": i, "vector": emb_text(line), "text": line})

In [ ]:
insert_res = milvus_client.insert(collection_name=collection_name, data=data)

### **4. Retrieve a data for query**

In [ ]:
def retrieve_context(question):
    embedding = emb_text(question)
    res = milvus_client.search(
        collection_name="mental_health_rag",
        data=[embedding],
        limit=3,
        search_params={"metric_type": "IP", "params": {}},
        output_fields=["text"]
    )

    strings = []
    for i in range(len(res[0])):
        strings.append(res[0][i]['entity']['text'])
        context = " ".join(strings)
    return context #" ".join(hit["entity"]["text"] for hit in res[0])

### **5. Hugging Face Pretrained Model**

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

qa_pipeline = pipeline("text2text-generation", model=model,tokenizer=tokenizer)

In [ ]:
def ask_bot(question):
    context = retrieve_context(question)

    PROMPT = """
    Use the information enclosed in <context> tags to provide an answer to the
    question enclosed in <question> tags.
    <context>
    {context}
    </context>
    <question>
    {question}
    </question>
    """
    prompt = PROMPT.format(context=context, question=question)


    result = qa_pipeline(prompt, max_new_tokens=256)[0]['generated_text']
    return result.strip()

In [ ]:
question = "How are Personality Disorders categorized, and what are the challenges in diagnosing them?​"

In [ ]:
retrieve_context(question)

In [ ]:
ask_bot(question)

In [ ]:
pip install gradio

In [ ]:
import gradio as gr
import time

def chatbot_interface(message, history):
    history.append((message, None))
    time.sleep(0.5)
    response = ask_bot(message)
    history[-1] = (message, response)
    return "", history

# JavaScript: animate title + show welcome message
js = """
function createGradioAnimation() {
    var container = document.createElement('div');
    container.id = 'gradio-animation';
    container.style.fontSize = '2em';
    container.style.fontWeight = 'bold';
    container.style.textAlign = 'center';
    container.style.marginBottom = '20px';

    var text = 'Hello! I am your Mental Health Assistant';
    for (var i = 0; i < text.length; i++) {
        (function(i){
            setTimeout(function(){
                var letter = document.createElement('span');
                letter.style.opacity = '0';
                letter.style.transition = 'opacity 0.5s';
                letter.innerText = text[i];

                container.appendChild(letter);

                setTimeout(function() {
                    letter.style.opacity = '1';
                }, 50);
            }, i * 250);
        })(i);
    }

    var gradioContainer = document.querySelector('.gradio-container');
    gradioContainer.insertBefore(container, gradioContainer.firstChild);

    return 'Animation created';
}
"""

# App UI
with gr.Blocks(js=js, css="""
body {
    background-color: #3e2723;
    color: #fefefe;
    font-family: "Helvetica Neue", Helvetica, Arial, sans-serif;
}
.gradio-container {
    max-width: 750px;
    margin: auto;
    padding: 20px;
}
.gr-chatbot {
    background-color: #ffffff;
    border-radius: 12px;
    width: 400px;
    padding: 10px;
    border: 1px solid #d1d5db;
}
.message.user {
    background-color: #c7d2fe;
    color: #111827;
    font-size: 1rem;
}
.message.bot {
    background-color: #e0f2fe;
    color: #111827;
    font-size: 1rem;
}
.gr-textbox {
    border-radius: 10px;
    border: 1px solid #d1d5db;
    background-color: #ffffff;
    color: #111827;
    font-size: 1rem;
}
.gr-button {
    font-size: 1rem;
    font-weight: 500;
}
#submit-btn {
    background-color: #6d4c41 !important;
    color: white !important;
    border-radius: 10px;
    padding: 10px 20px;
}
#clear-btn {
    background-color: #a1887f !important;
    color: white !important;
    border-radius: 10px;
    padding: 10px 20px;
    margin-right: 10px;
}
""") as demo:

    gr.Markdown("""
        <div style="text-align: center; margin-bottom: 1rem;">
            <p style="font-size: 0.9rem; color: #374151;">Ask your questions and receive informative, compassionate answers.</p>
        </div>
    """)

    chatbot = gr.Chatbot(elem_id="chatbot", label=None)
    state = gr.State([])

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Ask something like 'What is bipolar disorder?'",
            show_label=False,
            lines=1,
            container=True
        )

    with gr.Row():
        clear = gr.Button("Clear Chat", elem_id="clear-btn")
        submit = gr.Button("Send", elem_id="submit-btn")

    submit.click(chatbot_interface, [msg, state], [msg, chatbot])
    msg.submit(chatbot_interface, [msg, state], [msg, chatbot])
    clear.click(lambda: ([], []), outputs=[chatbot, state])

demo.launch()
